In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
gas = pd.read_excel(
    "../data/raw/henry_hub_daily.xls",
    sheet_name="Data 1",
    skiprows=2
)

gas.head()

,Date,Henry Hub Natural Gas Spot Price (Dollars per Million Btu)
0,1997-01-07,3.82
1,1997-01-08,3.80
2,1997-01-09,3.61
3,1997-01-10,3.92
4,1997-01-13,4.00


In [3]:
gas.columns

Index(['Date', 'Henry Hub Natural Gas Spot Price (Dollars per Million Btu)'], dtype='str')

In [4]:
gas.isnull().sum()

Date                                                          0
Henry Hub Natural Gas Spot Price (Dollars per Million Btu)    1
dtype: int64

In [5]:
gas.duplicated().sum()

np.int64(0)

In [6]:
# Rename columns
gas.columns = ["date", "gas_price"]

# Convert to correct data types
gas["date"] = pd.to_datetime(gas["date"])

gas["gas_price"] = pd.to_numeric(
    gas["gas_price"],
    errors="coerce"
)

# Remove missing prices
gas = gas.dropna()

# Sort chronologically
gas = gas.sort_values("date")

gas.head()

,date,gas_price
0,1997-01-07,3.82
1,1997-01-08,3.80
2,1997-01-09,3.61
3,1997-01-10,3.92
4,1997-01-13,4.00


In [7]:
gas.shape

(7446, 2)

In [8]:
gas.dtypes

date         datetime64[us]
gas_price           float64
dtype: object

In [9]:
gas.describe()

,date,gas_price
count,7446,7446.000000
mean,2011-11-09 11:26:32.586623,4.073163
min,1997-01-07 00:00:00,1.050000
25%,2004-06-16 06:00:00,2.610000
50%,2011-11-22 12:00:00,3.340000
75%,2019-03-21 18:00:00,4.990000
max,2026-09-01 00:00:00,30.720000
std,NaN,2.176470


In [10]:
gas = gas[
    (gas["date"] >= "2017-01-01") &
    (gas["date"] <= "2025-12-31")
].copy()

print(gas.shape)
print(gas["date"].min())
print(gas["date"].max())

(2258, 2)
2017-01-02 00:00:00
2025-12-31 00:00:00


In [11]:
gas.to_csv(
    "../data/processed/henry_hub_clean.csv",
    index=False
)

In [12]:
import requests

for year in range(2017, 2026):
    url = f"https://www.eia.gov/electricity/wholesale/xls/archive/ice_electric-{year}final.xlsx"
    
    r = requests.get(url)
    
    with open(f"../data/raw/electricity_{year}.xlsx", "wb") as f:
        f.write(r.content)

    print(year)

2017
2018
2019
2020
2021
2022
2023
2024
2025


In [13]:
import pandas as pd

all_pjm = []

for year in range(2017, 2026):
    df = pd.read_excel(f"../data/raw/electricity_{year}.xlsx")

    df = df[df["Price hub"] == "PJM WH Real Time Peak"]

    df = df[
        ["Trade date", "Wtd avg price $/MWh", "Daily volume MWh"]
    ]

    all_pjm.append(df)

pjm = pd.concat(all_pjm)

In [14]:
pjm.shape

(2194, 3)

In [15]:
pjm.columns = ["date", "power_price", "volume_mwh"]

pjm["date"] = pd.to_datetime(pjm["date"])
pjm["power_price"] = pd.to_numeric(pjm["power_price"], errors="coerce")
pjm["volume_mwh"] = pd.to_numeric(pjm["volume_mwh"], errors="coerce")

pjm = pjm.dropna().sort_values("date")

In [16]:
print(pjm.shape)
print(pjm["date"].min())
print(pjm["date"].max())

(2185, 3)
2017-01-03 00:00:00
2025-12-23 00:00:00


In [17]:
data = pd.merge(
    pjm,
    gas,
    on="date",
    how="inner"
)

data.head()

,date,power_price,volume_mwh,gas_price
0,2017-01-03,34.39,36000.0,3.41
1,2017-01-04,38.59,41600.0,3.42
2,2017-01-05,45.24,56000.0,3.42
3,2017-01-06,50.89,100000.0,3.38
4,2017-01-09,40.17,60800.0,3.14


In [18]:
print(data.shape)
print(data["date"].min())
print(data["date"].max())

(2149, 4)
2017-01-03 00:00:00
2025-12-23 00:00:00


In [19]:
data.to_csv("../data/processed/energy_market_data.csv", index=False)